# 2장 02. Precision과 Recall 직접 계산

이 notebook은 TP, FP, FN을 직접 세고 Precision과 Recall을 계산합니다. Accuracy 하나만 보지 않기 위한 최소 실습입니다.


## 기본 개념: Confusion Matrix와 오류 유형

Confusion Matrix는 label과 prediction의 조합을 세는 표입니다. 단순한 표처럼 보이지만, MLOps에서는 배포 후보 모델을 비교하고 threshold 정책을 설명하는 핵심 근거가 됩니다.

Precision은 Positive로 예측한 것 중 실제 Positive인 비율입니다. FP가 많아지면 Precision이 낮아집니다. Recall은 실제 Positive 중 모델이 찾아낸 비율입니다. FN이 많아지면 Recall이 낮아집니다.

Accuracy는 전체 중 맞힌 비율이라 이해하기 쉽지만, class 분포가 치우친 문제에서는 위험합니다. Accuracy가 비슷해도 FP가 늘어난 모델과 FN이 늘어난 모델은 운영 영향이 다릅니다.

| 값 | 계산 의미 | MLOps에서 쓰는 이유 |
| --- | --- | --- |
| TP | 실제 `high_risk`, 예측 `high_risk` | 탐지한 Positive 수 |
| FP | 실제 `low_risk`, 예측 `high_risk` | 불필요한 알림 또는 후속 처리 증가 |
| FN | 실제 `high_risk`, 예측 `low_risk` | 놓친 Positive 수 |
| Precision | TP / (TP + FP) | Positive 예측의 신뢰도 |
| Recall | TP / (TP + FN) | Positive class 탐지 범위 |

이 노트북에서는 복잡한 metric library를 쓰지 않고 직접 셉니다. 직접 세어 보면 모델 평가 기록, MLflow metric, Grafana dashboard에서 보이는 숫자가 어떤 분모와 분자에서 나오는지 이해할 수 있습니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
dataframe, source = aiq.load_csv_or_sample(
    "data/vital_signs_test.csv",
    aiq.sample_vital_signs(),
    nrows=30,
)
work = dataframe.copy()
work["label"] = work["label"].map(aiq.normalize_label)
work["score"] = aiq.score_rows(work)
work["prediction"] = "low_risk"
work.loc[work["score"] >= aiq.THRESHOLD, "prediction"] = "high_risk"


## 1. TP, FP, FN 세기

`high_risk`를 Positive class로 두고 네 가지 개수를 직접 셉니다.


In [ ]:
tp = ((work["label"] == "high_risk") & (work["prediction"] == "high_risk")).sum()
fp = ((work["label"] == "low_risk") & (work["prediction"] == "high_risk")).sum()
fn = ((work["label"] == "high_risk") & (work["prediction"] == "low_risk")).sum()
tn = ((work["label"] == "low_risk") & (work["prediction"] == "low_risk")).sum()

pd.DataFrame([{"TP": tp, "FP": fp, "FN": fn, "TN": tn}])


## 2. Precision과 Recall 계산

Precision은 high_risk라고 예측한 것 중 실제 high_risk 비율입니다. Recall은 실제 high_risk 중 찾아낸 비율입니다.


In [ ]:
precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0

pd.DataFrame([
    {"metric": "precision", "value": round(precision, 4)},
    {"metric": "recall", "value": round(recall, 4)},
])
